In [2]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE
import os

df = pd.read_csv('C:/Users/shruti/Documents/credit-card-churn-prediction/data/processed/clean_churn_data.csv')
df.drop(columns=['Unnamed: 21'], inplace=True)

print("Shape:", df.shape)
df.head()

Shape: (10127, 20)


,Attrition_Flag,Customer_Age,Gender,Dependent_count,Education_Level,Marital_Status,Income_Category,Card_Category,Months_on_book,Total_Relationship_Count,Months_Inactive_12_mon,Contacts_Count_12_mon,Credit_Limit,Total_Revolving_Bal,Avg_Open_To_Buy,Total_Amt_Chng_Q4_Q1,Total_Trans_Amt,Total_Trans_Ct,Total_Ct_Chng_Q4_Q1,Avg_Utilization_Ratio
0,0,45,M,3,High School,Married,$60K - $80K,Blue,39,5,1,3,12691.0,777,11914.0,1.335,1144,42,1.625,0.061
1,0,49,F,5,Graduate,Single,Less than $40K,Blue,44,6,1,2,8256.0,864,7392.0,1.541,1291,33,3.714,0.105
2,0,51,M,3,Graduate,Married,$80K - $120K,Blue,36,4,1,0,3418.0,0,3418.0,2.594,1887,20,2.333,0.000
3,0,40,F,4,High School,Unknown,Less than $40K,Blue,34,3,4,1,3313.0,2517,796.0,1.405,1171,20,2.333,0.760
4,0,40,M,3,Uneducated,Married,$60K - $80K,Blue,21,5,1,0,4716.0,0,4716.0,2.175,816,28,2.500,0.000


In [3]:
# These are the categorical columns we need to encode
cat_cols = ['Gender', 'Education_Level', 'Marital_Status', 
            'Income_Category', 'Card_Category']

print("Unique values per categorical column:")
for col in cat_cols:
    print(f"  {col}: {df[col].unique()}")

Unique values per categorical column:
  Gender: ['M' 'F']
  Education_Level: ['High School' 'Graduate' 'Uneducated' 'Unknown' 'College' 'Post-Graduate'
 'Doctorate']
  Marital_Status: ['Married' 'Single' 'Unknown' 'Divorced']
  Income_Category: ['$60K - $80K' 'Less than $40K' '$80K - $120K' '$40K - $60K' '$120K +'
 'Unknown']
  Card_Category: ['Blue' 'Gold' 'Silver' 'Platinum']


In [4]:
le = LabelEncoder()

for col in cat_cols:
    df[col] = le.fit_transform(df[col])
    
print("Encoding done!")
df[cat_cols].head()

Encoding done!


,Gender,Education_Level,Marital_Status,Income_Category,Card_Category
0,1,3,1,2,0
1,0,2,2,4,0
2,1,2,1,3,0
3,0,3,3,4,0
4,1,5,1,2,0


In [5]:
# Avg_Open_To_Buy is 99% correlated with Credit_Limit — drop it
df.drop(columns=['Avg_Open_To_Buy'], inplace=True)

print("Remaining columns:", df.columns.tolist())
print("Shape:", df.shape)

Remaining columns: ['Attrition_Flag', 'Customer_Age', 'Gender', 'Dependent_count', 'Education_Level', 'Marital_Status', 'Income_Category', 'Card_Category', 'Months_on_book', 'Total_Relationship_Count', 'Months_Inactive_12_mon', 'Contacts_Count_12_mon', 'Credit_Limit', 'Total_Revolving_Bal', 'Total_Amt_Chng_Q4_Q1', 'Total_Trans_Amt', 'Total_Trans_Ct', 'Total_Ct_Chng_Q4_Q1', 'Avg_Utilization_Ratio']
Shape: (10127, 19)


In [6]:
X = df.drop(columns=['Attrition_Flag'])
y = df['Attrition_Flag']

print("X shape:", X.shape)
print("y shape:", y.shape)
print("\nClass distribution before SMOTE:")
print(y.value_counts())

X shape: (10127, 18)
y shape: (10127,)

Class distribution before SMOTE:
Attrition_Flag
0    8500
1    1627
Name: count, dtype: int64


In [7]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2, 
    random_state=42, 
    stratify=y  # keeps same churn ratio in both splits
)

print("Train size:", X_train.shape)
print("Test size:", X_test.shape)
print("\nTrain class distribution:")
print(y_train.value_counts())

Train size: (8101, 18)
Test size: (2026, 18)

Train class distribution:
Attrition_Flag
0    6799
1    1302
Name: count, dtype: int64


In [8]:
smote = SMOTE(random_state=42)
X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)

print("After SMOTE:")
print("X_train shape:", X_train_sm.shape)
print("Class distribution:", pd.Series(y_train_sm).value_counts())

After SMOTE:
X_train shape: (13598, 18)
Class distribution: Attrition_Flag
0    6799
1    6799
Name: count, dtype: int64


In [9]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train_sm)
X_test_scaled = scaler.transform(X_test)  # ONLY transform, never fit on test!

print("Scaling done!")
print("X_train_scaled shape:", X_train_scaled.shape)
print("X_test_scaled shape:", X_test_scaled.shape)

Scaling done!
X_train_scaled shape: (13598, 18)
X_test_scaled shape: (2026, 18)


In [10]:
# Save to data/ folder for use in modelling step
os.makedirs('../data/processed', exist_ok=True)

np.save('../data/processed/X_train.npy', X_train_scaled)
np.save('../data/processed/X_test.npy', X_test_scaled)
np.save('../data/processed/y_train.npy', y_train_sm)
np.save('../data/processed/y_test.npy', y_test)

# Save feature names for later use
feature_names = X.columns.tolist()
pd.Series(feature_names).to_csv('../data/processed/feature_names.csv', index=False)

print("All files saved to data/processed/")
print("Features:", feature_names)


All files saved to data/processed/
Features: ['Customer_Age', 'Gender', 'Dependent_count', 'Education_Level', 'Marital_Status', 'Income_Category', 'Card_Category', 'Months_on_book', 'Total_Relationship_Count', 'Months_Inactive_12_mon', 'Contacts_Count_12_mon', 'Credit_Limit', 'Total_Revolving_Bal', 'Total_Amt_Chng_Q4_Q1', 'Total_Trans_Amt', 'Total_Trans_Ct', 'Total_Ct_Chng_Q4_Q1', 'Avg_Utilization_Ratio']


## Summary

- Encoded 5 categorical columns using LabelEncoder
- Dropped `Avg_Open_To_Buy` (0.99 correlation with `Credit_Limit`)
- Split data: 80% train / 20% test (stratified)
- Applied SMOTE to training set → balanced classes
- Scaled all features with StandardScaler
- Saved processed arrays to `data/processed/`